# Session 3: Prompt Engineering and Retrieval-Augmented Generation Lab

**Course:** Language Models: ML Basics to Modern AI (BTU Cottbus, M.Sc. AI seminar)
**Session:** 3 of 4
**Lecture reference:** Lecture on alignment, prompting techniques (zero-shot, few-shot, chain-of-thought), and retrieval-augmented generation.

## Learning objectives

By the end of this notebook you should be able to:

- Explain the behavioural difference between a base language model and an instruction-tuned model on the same prompt.
- Apply zero-shot, few-shot, and chain-of-thought prompting and describe the trade-offs of each.
- Build a working retrieval-augmented generation (RAG) pipeline from scratch: chunk a corpus, embed the chunks, retrieve top-k by cosine similarity, and stuff the retrieved context into a prompt.
- Identify when retrieval improves model factuality on questions outside the pretraining distribution.
- Read and adapt the four-step RAG recipe to a new corpus and a new question set.

The notebook accompanies the lecture on alignment and prompting. It uses a small instruction-tuned model (SmolLM2-135M-Instruct, ~135 M parameters) so every generation completes within a few seconds on a free Colab CPU.


## §1 Primer: in-context learning and retrieval

This primer collects the two ideas you will exercise in §3 and §5: shaping model behaviour through prompts (in-context learning) and grounding model outputs in retrieved facts (retrieval-augmented generation). Both techniques operate at inference time. Neither updates a single weight.

### §1a In-context learning

Imagine reading the first three lines of a poem you have never seen and being asked to predict the fourth line. You would lean on the rhyme scheme, the meter, and the imagery already on the page. A pretrained language model does something similar: it has spent training on the task of continuing patterns it sees in its input, and it brings that habit to bear on whatever prefix you hand it. In-context learning is the practice of arranging that prefix so the continuation the model produces is the answer you want.

The plainest form is **zero-shot prompting**: state the task in natural language and ask for the answer. "Classify the sentiment of this sentence as positive, negative, or neutral." Instruction-tuned models are trained on millions of such request-and-response pairs, so they tend to follow the instruction directly. A pure base model that has only seen raw text instead tries to continue the text as if it were a corpus sample, often by writing more sentences in a similar style rather than producing the requested classification. The §3 Demo A puts both behaviours side by side on the same prompt.

**Few-shot prompting** sharpens the picture by supplying a handful of worked examples in the prefix. Each example has the same input-output structure you want the model to produce on the test query, so the prefix looks like a small, internally consistent task:

```
Sentence: The movie was breathtaking.
Sentiment: positive

Sentence: I waited two hours and the food was cold.
Sentiment: negative

Sentence: The presenter spoke clearly but the slides were unreadable.
Sentiment:
```

Continuing this pattern means producing the missing label. The model has not been retrained; it has been shown the input-output schema explicitly. Few-shot reliability tends to climb sharply with the first few examples and then plateau. Three to five examples is typical for classification-style tasks.

**Chain-of-thought (CoT) prompting** factors a multi-step task into intermediate steps written out as text. Instead of asking "What is the answer?" you ask "Let us work through this step by step." The model then writes its reasoning trace before committing to a final answer. The interesting effect is that the trace becomes a scratchpad: each step is a piece of context that the next forward pass conditions on, so the model can build up partial computations one token at a time rather than having to produce the final answer in one shot. On arithmetic, logic puzzles, and multi-hop questions this produces meaningfully more correct answers, especially on models too small to solve the task directly.

Why does any of this work? Pretraining exposes the model to a wide distribution of text that includes problem-solution pairs, worked examples, and step-by-step reasoning. A prompt that puts the model into a distribution it has seen often steers its next-token predictions toward the kind of completion that distribution contains. Few-shot and CoT are practical ways to put the model into a known-good distribution at inference time without touching its weights. The empirical scaling results presented in the lecture (the abrupt emergence of strong few-shot performance at certain model sizes) are the quantitative side of this story; the qualitative side is just that arranging a good prefix is a real lever on model behaviour.

### §1b Retrieval-augmented generation

A pretrained language model knows what was in its training corpus. It does not know what was published after the data cutoff, what is in your private documentation, or what your company decided last Tuesday. Asking it questions about any of that produces confident-sounding answers that may be entirely fabricated. Retrieval-augmented generation (RAG) addresses this gap by giving the model the relevant facts in its prompt, fetched from an external store at query time.

The intuition is small. If you wanted to answer "How many people work at BlueQuokka?" and you happened to have a one-paragraph handbook entry that says "BlueQuokka employs 167 people across three sites," you would just read that paragraph and answer from it. RAG automates that lookup: store the handbook somewhere searchable, find the most relevant paragraph for each new question, paste it into the prompt, and let the model answer from the stuffed context.

The standard pipeline is four steps. **Chunk** the corpus into pieces small enough to fit inside the model's context window, with each piece self-contained enough to be useful on its own. **Embed** each chunk into a fixed-length vector using a sentence-embedding model; the vector captures the chunk's meaning rather than its exact wording. **Search** at query time by embedding the question with the same model and finding the chunks whose vectors are closest to the question vector under cosine similarity. **Stuff** the top-k retrieved chunks into the prompt as a context block, followed by the question, and ask the model to answer using that context.

Dense embedding search is what makes the retrieval step work for questions that do not match the corpus wording. A keyword index would miss the chunk "BlueQuokka employs 167 people" if the question asked "How big is the BlueQuokka workforce" because none of the question's content words appear in the chunk. The two pieces of text live near each other in embedding space because the embedding model was trained to map semantically related text to similar vectors, so the cosine score is high even though the surface forms differ. That semantic robustness is the main advantage over a pure keyword search.

"Stuffing" the prompt is exactly what it sounds like. The prompt is just a string of tokens, and the model has no special channel for retrieved context. You write the context into the string before the question, label it clearly so the model can find the relevant facts, and ask the question after it. A typical layout is "Use the following context to answer the question. Context: [chunk 1] [chunk 2] [chunk 3]. Question: [the question]. Answer:" and the model continues from "Answer:" by reading the context you provided.

The §5 build applies all four steps to a 20-paragraph fictional handbook for BlueQuokka, Inc. The handbook contains specific facts that the instruction model has never seen in pretraining. Without retrieval the model invents plausible-sounding answers; with retrieval it can quote the right paragraph. The pipeline lifts directly to a real corpus by swapping the chunk source.


## §2 Setup

The cell below loads two language models and one sentence embedder. The base model is GPT-2 (already cached from earlier sessions) and serves as the "no instructions" comparison in §3 Demo A. The instruction-tuned model is SmolLM2-135M-Instruct, used for every other generation in this notebook. The embedder is `all-MiniLM-L6-v2`, a 384-dim sentence-embedding model used for retrieval in §5. All three downloads happen once and are cached for subsequent runs.


In [ ]:
"""§2 Setup: imports, seed, device, models, embedder."""

import math
import random
from copy import deepcopy
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import HTML, display
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {DEVICE}, torch: {torch.__version__}")

# Base model for the "no instructions" comparison.
base_tokenizer = AutoTokenizer.from_pretrained("gpt2")
base_model = AutoModelForCausalLM.from_pretrained("gpt2").to(DEVICE).eval()

# Instruction-tuned model for everything else.
INSTRUCT_NAME = "HuggingFaceTB/SmolLM2-135M-Instruct"
instruct_tokenizer = AutoTokenizer.from_pretrained(INSTRUCT_NAME)
instruct_model = AutoModelForCausalLM.from_pretrained(INSTRUCT_NAME).to(DEVICE).eval()

# Sentence embedder for retrieval (loaded only when needed below, since it pulls
# extra dependencies on first use).
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=str(DEVICE))

print(f"base model:     gpt2 ({sum(p.numel() for p in base_model.parameters()):,} params)")
print(f"instruct model: {INSTRUCT_NAME} ({sum(p.numel() for p in instruct_model.parameters()):,} params)")
print(f"embedder:       all-MiniLM-L6-v2 (dim {embedder.get_sentence_embedding_dimension()})")


## §3 Guided exploration: three classical prompting techniques

The three code cells below run static demos that put the techniques from §1a into practice. Demo A sends the same prompt to a base model (GPT-2) and an instruction-tuned model (SmolLM2-Instruct) so you can see what instruction tuning does to model behaviour. Demo B compares zero-shot and few-shot sentiment classification on the same query. Demo C contrasts a direct prompt with a chain-of-thought prompt on a small arithmetic word problem. Each demo runs once at notebook load and prints both outputs in a two-column comparison table.


In [ ]:
"""§3 Demo A: base GPT-2 vs SmolLM2-Instruct on the same prompt."""

PROMPT_A = "Write a one-sentence summary of how a transformer learns to predict the next token."


def _generate(model, tokenizer, prompt: str, max_new_tokens: int = 60) -> str:
    ids = tokenizer.encode(prompt, return_tensors="pt").to(DEVICE)
    attention_mask = torch.ones_like(ids)
    with torch.no_grad():
        out = model.generate(
            ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)


def _generate_instruct(prompt: str, max_new_tokens: int = 60) -> str:
    """Wrap the prompt in the SmolLM2 chat template."""
    messages = [{"role": "user", "content": prompt}]
    text = instruct_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    ids = instruct_tokenizer.encode(text, return_tensors="pt").to(DEVICE)
    attention_mask = torch.ones_like(ids)
    with torch.no_grad():
        out = instruct_model.generate(
            ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            pad_token_id=instruct_tokenizer.eos_token_id,
        )
    return instruct_tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)


base_out = _generate(base_model, base_tokenizer, PROMPT_A)
instruct_out = _generate_instruct(PROMPT_A)

print(f"Prompt: {PROMPT_A!r}\n")

def _two_col(left_title: str, left: str, right_title: str, right: str) -> str:
    cell_css = (
        "vertical-align:top; padding:10px; border:1px solid #cbd5e1; "
        "width:50%; font-family:Georgia, serif; line-height:1.5;"
    )
    return (
        "<table style='border-collapse:collapse; width:100%;'>"
        f"<tr><th style='{cell_css} background:#f1f5f9;'>{left_title}</th>"
        f"<th style='{cell_css} background:#dbeafe;'>{right_title}</th></tr>"
        f"<tr><td style='{cell_css}'>{left}</td>"
        f"<td style='{cell_css}'>{right}</td></tr></table>"
    )


display(HTML(_two_col("gpt2 (base)", base_out, "SmolLM2-Instruct", instruct_out)))
print("\nBase output (plain):", base_out)
print("\nInstruct output (plain):", instruct_out)


In [ ]:
"""§3 Demo B: zero-shot vs few-shot sentiment classification."""

TASK_B = "Classify the sentiment of the following sentence as positive, negative, or neutral."

EXAMPLES_B = [
    ("The movie was breathtaking and moved me to tears.", "positive"),
    ("I waited two hours and the food arrived cold.", "negative"),
    ("The book is exactly as advertised, no surprises.", "neutral"),
]
QUERY_B = "The presenter spoke clearly but the slides were unreadable."

# Zero-shot.
zero_shot_prompt = f"{TASK_B}\n\nSentence: {QUERY_B}\nSentiment:"
zero_shot_out = _generate_instruct(zero_shot_prompt, max_new_tokens=8)

# Few-shot.
few_shot_examples = "\n\n".join(
    f"Sentence: {s}\nSentiment: {label}" for s, label in EXAMPLES_B
)
few_shot_prompt = f"{TASK_B}\n\n{few_shot_examples}\n\nSentence: {QUERY_B}\nSentiment:"
few_shot_out = _generate_instruct(few_shot_prompt, max_new_tokens=8)

print(f"Query: {QUERY_B!r}\n")
display(HTML(_two_col("Zero-shot", zero_shot_out, "Few-shot (3 examples)", few_shot_out)))
print("\nZero-shot output:", zero_shot_out)
print("Few-shot output:", few_shot_out)


In [ ]:
"""§3 Demo C: direct answer vs chain-of-thought on a math word problem."""

PROBLEM_C = (
    "Anna has 17 apples. She gives 4 to Ben and then buys 9 more. "
    "Ben eats 2 of his apples and gives the rest to Carla. "
    "How many apples does each person have at the end?"
)

direct_prompt = f"{PROBLEM_C}\n\nAnswer:"
direct_out = _generate_instruct(direct_prompt, max_new_tokens=80)

cot_prompt = f"{PROBLEM_C}\n\nLet's work through this step by step.\n"
cot_out = _generate_instruct(cot_prompt, max_new_tokens=180)

print(f"Problem: {PROBLEM_C!r}\n")
display(HTML(_two_col("Direct answer", direct_out, "Chain-of-thought", cot_out)))
print("\nDirect output:", direct_out)
print("\nCoT output:", cot_out)


## §4 Warm-ups

Two short exercises before the deep build. Warm-up 1 builds a reusable few-shot prompt template that you can apply to any classification-style task. Warm-up 2 implements the cosine-similarity top-k primitive that the retrieval step of the RAG pipeline depends on.


In [ ]:
"""§4 Warm-up 1 (exercise): few-shot prompt template.

Implement `build_few_shot_prompt` so that given a list of (input, output)
example pairs, a final query, and a one-line task description, it returns a
single prompt string. The structure should be: the task description, then each
example as 'Input: ...' on one line and 'Output: ...' on the next, separated
by blank lines, then a final 'Input: <query>' followed by 'Output:' with no
value (so the model continues from there).
"""


def build_few_shot_prompt(
    examples: list[tuple[str, str]],
    query: str,
    task_description: str,
    input_label: str = "Input",
    output_label: str = "Output",
) -> str:
    """Format a list of (input, output) examples plus a final query into a single prompt."""
    # TODO: assemble the string described in the docstring. Walk the examples
    # list, format each (input, output) pair with the labels, separate the
    # blocks with blank lines, and end the prompt with the query followed by
    # the output label and a colon.
    raise NotImplementedError


_examples = [("happy", "positive"), ("sad", "negative"), ("ok", "neutral")]
_query = "excited"
try:
    print(build_few_shot_prompt(_examples, _query, "Classify the emotion."))
except NotImplementedError:
    print("build_few_shot_prompt not implemented yet.")


In [ ]:
"""§4 Warm-up 2 (exercise): cosine-similarity top-k search.

Implement `cosine_topk(query_emb, doc_embs, k)` that returns the top-k document
indices and their cosine-similarity scores against the query. Both inputs are
numpy arrays. L2-normalise both first, then cosine reduces to a dot product.
Argsort by score in descending order and slice the first k results.
"""

import numpy as np


def cosine_topk(query_emb: np.ndarray, doc_embs: np.ndarray, k: int = 3) -> tuple[np.ndarray, np.ndarray]:
    """Return (top_indices, top_scores) sorted by cosine similarity to query_emb."""
    # TODO: normalise the query embedding to unit length, normalise each row of
    # the doc embedding matrix, compute the dot product to obtain a score per
    # document, argsort in descending order, return the first k indices and
    # the matching scores.
    raise NotImplementedError


_q = np.array([1.0, 0.0])
_d = np.array([[1.0, 0.0], [0.7, 0.7], [0.0, 1.0], [-1.0, 0.0]])
try:
    _idx, _scores = cosine_topk(_q, _d, k=3)
    print("top indices:", _idx.tolist())
    print("top scores:", _scores.tolist())
    # Expected: top indices [0, 1, 2] with scores [1.0, 0.707, 0.0].
except NotImplementedError:
    print("cosine_topk not implemented yet.")


## §5 Deep build: RAG on the BlueQuokka handbook

Five subtasks build a retrieval-augmented generation pipeline from scratch on a hand-authored fictional company handbook. BlueQuokka, Inc. is a made-up Australian-German tablet manufacturer; the 20 paragraphs below contain specific facts about its founding, products, sites, headcount, finances, and internal policies. The instruction model has never seen any of this in pretraining, so without retrieval its answers about BlueQuokka are confabulated. With retrieval the model can quote the handbook directly.

The subtasks are: (1) load and chunk the corpus, (2) embed every chunk, (3) build a top-k retriever, (4) build a context-stuffed prompt, (5) run three factual questions with and without retrieval and compare the answers.


In [ ]:
"""§5 Subtask 1 (exercise): load and chunk the corpus.

The CORPUS list below is the BlueQuokka handbook: 20 short paragraphs, each
self-contained. Your job is to turn this into the list of strings the embedder
will work on. For paragraphs that are already short (30-80 words), one chunk
per paragraph is the natural unit. For longer documents you would split on
tokens or sentences and might overlap consecutive windows.
"""

CORPUS = [
    "BlueQuokka, Inc. was founded in 2017 in Adelaide, Australia by Maria Henson "
    "and Tomás Riesgo. The original product was a tablet-based field-research "
    "logger for marsupial biologists, branded the BQ-1.",

    "The company moved its headquarters to Hamburg, Germany in 2020 to be closer "
    "to its European customer base. The Adelaide office remained open as the "
    "primary research-and-development site, currently employing 38 people.",

    "BlueQuokka's flagship product line is the BQ-3, a ruggedised tablet for "
    "remote-fieldwork data capture. The BQ-3 supports cellular fallback, runs on "
    "a customised Android 13 build, and ships with a six-week battery life under "
    "typical use.",

    "All BlueQuokka tablets ship with the QuokkaCollect software suite, "
    "developed in-house. QuokkaCollect supports offline form design, geotagged "
    "media capture, and synchronisation with both QuokkaCloud and self-hosted "
    "PostgreSQL endpoints.",

    "QuokkaCloud is BlueQuokka's hosted backend service, launched in 2021 and "
    "billed per active tablet per month. Enterprise customers can opt into a "
    "self-hosted deployment instead, which is the preferred configuration for "
    "government and academic clients.",

    "The company's annual revenue in fiscal year 2024 was 14.2 million Euro, "
    "up 31 percent from the previous year. About 62 percent of revenue came "
    "from hardware sales, with the remainder from software subscriptions and "
    "professional services.",

    "BlueQuokka employs 167 people across three sites: Hamburg (89, including "
    "all of executive leadership and most of the commercial team), Adelaide "
    "(38, primarily product engineering and field-test operations), and "
    "Wrocław, Poland (40, mostly software engineering and customer support).",

    "The company's customer base is concentrated in three sectors: academic "
    "field research (around 45 percent of accounts), environmental NGOs "
    "(around 35 percent), and government agencies focused on biodiversity "
    "monitoring (around 20 percent).",

    "BlueQuokka's most-cited deployment is the WWF Eastern Indonesian Reef "
    "Survey, where 240 BQ-3 tablets are used by a network of trained dive "
    "operators to log coral health and species observations. The deployment "
    "began in 2022 and is now in its fourth season.",

    "BlueQuokka's research-and-development budget for 2025 is targeted at "
    "around 22 percent of revenue. The largest in-flight project is the BQ-4, "
    "which adds an integrated bioacoustic recorder and is scheduled for "
    "release in the second quarter of 2026.",

    "The company's CTO, Mateusz Korba, joined in 2021 from a senior role at "
    "the European Space Agency. Korba oversees both the Adelaide and Wrocław "
    "engineering teams and reports directly to CEO Maria Henson.",

    "BlueQuokka holds 11 patents across hardware ruggedisation and offline-"
    "first synchronisation. The company's stated policy is to license its "
    "patents royalty-free to non-profit and academic users.",

    "BlueQuokka follows a four-day work week as standard since January 2024. "
    "Employees work Monday through Thursday with Friday as a paid non-working "
    "day. The change was introduced as part of a productivity-and-wellbeing "
    "pilot and was made permanent after a one-year evaluation.",

    "Internal travel between sites is supported by a fixed annual budget of "
    "4,800 Euro per employee. Engineering staff are expected to spend at "
    "least one week per year at a different site to maintain cross-site "
    "familiarity.",

    "The company runs a 12-week summer internship programme, hosting 14 "
    "interns in 2024 split across the three sites. About half of recent "
    "interns have accepted full-time offers, mostly in software engineering "
    "and customer success roles.",

    "BlueQuokka has committed publicly to net-zero operations by 2028. The "
    "current carbon footprint is dominated by hardware manufacturing in "
    "Shenzhen and by air travel between the Adelaide and Hamburg offices.",

    "The company's open-source repository contains the QuokkaCollect form-"
    "design DSL, released under the Apache 2.0 licence in 2023. As of "
    "mid-2025 the repository has 412 GitHub stars and accepts pull requests "
    "from external contributors.",

    "BlueQuokka does not sell hardware to military or defence-adjacent "
    "customers as a matter of policy. The policy was adopted in 2019 and is "
    "reviewed annually by the board.",

    "Customer support is handled out of Wrocław, with a stated four-hour "
    "response time for Enterprise tier accounts and a 24-hour response time "
    "for the standard tier. The support team operates from 06:00 to 22:00 "
    "Central European Time on weekdays.",

    "The next investor update is scheduled for September 2025 at the Hamburg "
    "office, with a roadmap session for QuokkaCloud's enterprise tier and a "
    "demonstration of the BQ-4 prototype.",
]

print(f"Corpus: {len(CORPUS)} paragraphs, {sum(len(p.split()) for p in CORPUS)} words total.")
print("\nFirst paragraph:")
print(CORPUS[0])

# TODO: build a list called `chunks` of strings to embed. For paragraphs that
# are already short (30-80 words each), one chunk per paragraph is the natural
# unit; for longer documents you would split on tokens or sentences and may
# choose to overlap windows.
chunks = []  # replace

if chunks:
    print(f"\nChunks ready: {len(chunks)}.")
else:
    print("\nchunks is empty; downstream cells will skip until you populate it.")


In [ ]:
"""§5 Subtask 2: embed every chunk with sentence-transformers."""

import numpy as np

if chunks:
    chunk_embs = embedder.encode(chunks, convert_to_numpy=True, show_progress_bar=False)
    chunk_embs = chunk_embs / (np.linalg.norm(chunk_embs, axis=1, keepdims=True) + 1e-12)
    print(f"Chunk embeddings shape: {chunk_embs.shape}")
    print(f"All unit norm: {np.allclose(np.linalg.norm(chunk_embs, axis=1), 1.0)}")
else:
    chunk_embs = None
    print("Skipping embedding: chunks is empty. Complete Subtask 1 first.")


In [ ]:
"""§5 Subtask 3 (exercise): top-k retrieval by cosine similarity.

Implement `retrieve(query, k)` that uses the already-loaded `embedder` and the
already-normalised `chunk_embs` matrix to return the top-k chunks for any
question. The chunk matrix is unit-norm so cosine similarity reduces to a dot
product after you normalise the query embedding.
"""


def retrieve(query: str, k: int = 3) -> list[tuple[str, float]]:
    """Embed the query, score against every chunk, return top-k (chunk, score)."""
    # TODO: embed the query, L2-normalise it, dot it against the (already
    # normalised) chunk_embs matrix to get a score per chunk, return the
    # k highest-scoring (chunk, score) pairs.
    raise NotImplementedError


try:
    for chunk, score in retrieve("how many people work at BlueQuokka?", k=3):
        print(f"[{score:.3f}] {chunk[:120]}...")
except NotImplementedError:
    print("retrieve not implemented yet.")
except (NameError, TypeError) as err:
    print(f"retrieve could not run: {type(err).__name__}: {err}")


In [ ]:
"""§5 Subtask 4 (exercise): build a RAG-stuffed prompt.

Implement `build_rag_prompt(question, retrieved)` that returns a single prompt
string containing the question, the retrieved chunks as context, and a final
'Answer:' tag so the model continues from there. A robust format states the
question, then labels the context block clearly (each chunk prefixed with
something like '[doc 1]'), then asks the model to answer using the context.
For small instruction models, repeating the question after the context helps
keep the model focused.
"""


def build_rag_prompt(question: str, retrieved: list[tuple[str, float]]) -> str:
    """Format the retrieved chunks into a context-then-question prompt."""
    # TODO: assemble a single string. Include the question, a context block
    # listing each retrieved chunk with a short tag (e.g. '[doc 1]'), an
    # instruction to use the context, and end with 'Answer:' so the model
    # continues from there. Consider repeating the question after the context
    # so the model does not lose track of what it is being asked.
    raise NotImplementedError


try:
    _example = build_rag_prompt(
        "how many people work at BlueQuokka?",
        retrieve("how many people work at BlueQuokka?"),
    )
    print(_example)
except NotImplementedError:
    print("build_rag_prompt not implemented yet.")
except (NameError, TypeError) as err:
    print(f"build_rag_prompt could not run: {type(err).__name__}: {err}")


In [ ]:
"""§5 Subtask 5 (exercise): with-vs-without retrieval on three handbook questions.

For each question in QUESTIONS, generate two answers using `_generate_instruct`
(defined in §3 Demo A): one with the bare question and one with the
RAG-stuffed prompt. Display both answers side by side using the `_two_col_q`
HTML helper below.
"""

QUESTIONS = [
    "What is BlueQuokka's flagship product, and what are its key features?",
    "How is BlueQuokka structured across its three sites, and how many people work at each?",
    "What is BlueQuokka's policy on selling to military customers, and when was it adopted?",
]


def _two_col_q(question: str, no_rag: str, rag: str) -> str:
    cell_css = (
        "vertical-align:top; padding:10px; border:1px solid #cbd5e1; "
        "width:50%; font-family:Georgia, serif; line-height:1.5;"
    )
    return (
        f"<p style='font-weight:600;margin:12px 0 4px;'>{question}</p>"
        "<table style='border-collapse:collapse; width:100%;'>"
        f"<tr><th style='{cell_css} background:#fef3c7;'>No retrieval</th>"
        f"<th style='{cell_css} background:#dcfce7;'>With RAG</th></tr>"
        f"<tr><td style='{cell_css}'>{no_rag}</td>"
        f"<td style='{cell_css}'>{rag}</td></tr></table>"
    )


# Guard against unimplemented retrieve / build_rag_prompt: try once on the
# first question; if that raises NotImplementedError, skip the loop with a
# clear message.
_rag_ready = True
try:
    _probe = build_rag_prompt(QUESTIONS[0], retrieve(QUESTIONS[0], k=3))
except (NotImplementedError, NameError, TypeError):
    _rag_ready = False

if not _rag_ready:
    print("Skipping the comparison: complete Subtasks 3 and 4 first so retrieve and build_rag_prompt are wired up.")
else:
    for q in QUESTIONS:
        # TODO: call _generate_instruct on the bare question to get the
        # no-retrieval answer. Then call retrieve and build_rag_prompt to
        # assemble the RAG prompt, and call _generate_instruct on that prompt
        # to get the RAG answer. Display both side by side using _two_col_q
        # and also print them so the comparison shows up in plain text.
        pass


## §6 Recap and next step

You have exercised the two inference-time levers covered by the lecture. Prompting (zero-shot, few-shot, chain-of-thought) shapes model behaviour by arranging the prefix to put the model into a distribution where its next-token predictions match the task you want. Retrieval-augmented generation extends what the model can answer correctly by fetching the relevant facts from an external store and stuffing them into the prompt at query time. Both techniques operate on the same fixed weights; both are productive when you do not have the budget or the data to retrain.

The next notebook, Session 4, builds an agent: a loop in which the model decides its own next prompt by reading tool outputs and writing the next tool call.
